In [7]:
# Cell 1: Install Libraries
%pip install transformers datasets scikit-learn accelerate -q

In [2]:
# Cell 2: Upload Your CSV Files
from google.colab import files
import os

# A list of the files we expect
expected_files = ['benign.csv', 'portscan.csv', 'dos.csv', 'brute_force.csv']
files_uploaded = []

for f in expected_files:
    if not os.path.exists(f):
        print(f"Please upload: {f}")
        uploaded = files.upload()
        if f in uploaded:
            files_uploaded.append(f)
        else:
            print(f"Error: Did not find {f} in upload.")
    else:
        print(f"{f} already exists. Skipping upload.")
        files_uploaded.append(f)

if len(files_uploaded) == len(expected_files):
    print("\n✅ All four files are present! You can proceed to the next cell.")
else:
    print(f"\n❌ Missing files. You have: {files_uploaded}")

Please upload: benign.csv


Saving benign.csv to benign.csv
Please upload: portscan.csv


Saving portscan.csv to portscan.csv
Please upload: dos.csv


Saving dos.csv to dos.csv
Please upload: brute_force.csv


Saving brute_force.csv to brute_force.csv

✅ All four files are present! You can proceed to the next cell.


In [8]:
# Cell 3: Load, Label, and "Sentence-ize" Data
import pandas as pd
import numpy as np

# --- 1. Load the CSVs ---
try:
    df_benign = pd.read_csv("benign.csv")
    df_portscan = pd.read_csv("portscan.csv")
    df_dos = pd.read_csv("dos.csv")
    df_brute = pd.read_csv("brute_force.csv")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please make sure all 4 files were uploaded correctly in Cell 2.")

# --- 2. Clean and Label them ---
dfs = {
    "BENIGN": df_benign,
    "PORTSCAN": df_portscan,
    "DOS": df_dos,
    "BRUTE_FORCE": df_brute
}

df_list = []
for label, df in dfs.items():
    # Drop any malformed/empty rows from cicflowmeter
    df.dropna(inplace=True)
    df.columns = df.columns.str.strip()
    df['Label'] = label
    df_list.append(df)

# --- 3. Combine them into ONE training set ---
df_train = pd.concat(df_list, ignore_index=True)

# --- 4. Preprocess for SLM ---
# Clean inf and NaN (cicflowmeter can produce these)
df_train.replace([np.inf, -np.inf], np.nan, inplace=True)
df_train.dropna(inplace=True)

# Define the feature columns (all columns EXCEPT the ones we drop)
drop_cols = ['Label', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'timestamp']
feature_cols = [col for col in df_train.columns if col not in drop_cols]

print(f"Using {len(feature_cols)} features for sentence conversion.")

# --- 5. "Sentence-ize" the Data ---
def convert_row_to_text(row):
    features = []
    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(row[col]):
            # Using int() makes the "word" cleaner (e.g., "fwd_packets:10")
            features.append(f"{col}:{int(row[col])}")
        else:
            features.append(f"{col}:{row[col]}")
    return " ".join(features)

print("Converting rows to text 'sentences'...")
df_train['text'] = df_train.apply(convert_row_to_text, axis=1)

# --- 6. Create the Final Dataset for the SLM ---
# We just need the 'text' and 'Label'
df_slm = df_train[['text', 'Label']].copy()

# --- 7. Create Label Mappings ---
# We need to convert string labels (e.g., "BENIGN") to numbers (e.g., 0)
labels = df_slm['Label'].unique()
labels.sort() # Ensure consistent order

id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

df_slm['labels'] = df_slm['Label'].map(label2id)

print("\n--- Label Distribution ---")
print(df_slm['Label'].value_counts())

print("\n--- Example of 'Sentences' for SLM ---")
print(df_slm.head())

print("\n--- Label Mappings ---")
print(f"ID to Label: {id2label}")
print(f"Label to ID: {label2id}")

Using 77 features for sentence conversion.
Converting rows to text 'sentences'...

--- Label Distribution ---
Label
BENIGN         1542
DOS            1317
BRUTE_FORCE    1268
PORTSCAN       1001
Name: count, dtype: int64

--- Example of 'Sentences' for SLM ---
                                                text   Label  labels
0  protocol:6 flow_duration:18.692754 flow_byts_s...  BENIGN       0
1  protocol:6 flow_duration:0.07153 flow_byts_s:1...  BENIGN       0
2  protocol:6 flow_duration:0.083671 flow_byts_s:...  BENIGN       0
3  protocol:6 flow_duration:29.970813 flow_byts_s...  BENIGN       0
4  protocol:6 flow_duration:0.086147 flow_byts_s:...  BENIGN       0

--- Label Mappings ---
ID to Label: {0: 'BENIGN', 1: 'BRUTE_FORCE', 2: 'DOS', 3: 'PORTSCAN'}
Label to ID: {'BENIGN': 0, 'BRUTE_FORCE': 1, 'DOS': 2, 'PORTSCAN': 3}


In [9]:
# Cell 4: Tokenize the Data
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

# --- 1. Split the Data ---
# We'll do a 80/20 train/test split
train_df, test_df = train_test_split(
    df_slm,
    test_size=0.2,
    random_state=42,
    stratify=df_slm['Label']
)

# --- 2. Convert to Hugging Face Dataset object ---
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print("\n--- Dataset Shapes ---")
print(f"Train dataset: {train_dataset.shape}")
print(f"Test dataset: {test_dataset.shape}")

# --- 3. Load Tokenizer ---
# We use 'distilbert-base-uncased' - a small, fast, and powerful SLM
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# --- 4. Create Tokenizing Function ---
def tokenize_function(examples):
    # 'truncation=True' will cut off sentences that are too long
    # 'padding="max_length"' will add padding to sentences that are too short
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# --- 5. Apply Tokenizer to Datasets ---
print("\nTokenizing datasets... (This may take a minute)")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

print("✅ Tokenization complete.")
print(tokenized_train_dataset)


--- Dataset Shapes ---
Train dataset: (4102, 3)
Test dataset: (1026, 3)

Tokenizing datasets... (This may take a minute)


Map:   0%|          | 0/4102 [00:00<?, ? examples/s]

Map:   0%|          | 0/1026 [00:00<?, ? examples/s]

✅ Tokenization complete.
Dataset({
    features: ['text', 'Label', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 4102
})


In [11]:
# Cell 5: Train the SLM (Legacy Version)
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# --- 1. Load the Model ---
# We tell the model how many labels to expect (4) and give it our mappings
num_labels = len(labels)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# --- 2. Define Metrics Function ---
# This function will be called during training to compute accuracy and F1
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Get the highest probability class
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')

    return {"accuracy": accuracy, "f1-macro": f1}

# --- 3. Set Training Arguments ---
# These are hyperparameters for the training process
# --- MODIFIED: Removed evaluation_strategy, save_strategy, and load_best_model_at_end ---
training_args = TrainingArguments(
    output_dir="slm_cyber_model",         # Directory to save the model
    learning_rate=2e-5,                  # Standard learning rate for fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,                  # 3 epochs is usually enough for fine-tuning
    weight_decay=0.01,
    push_to_hub=False,
    report_to="none",                    # Disables wandb
)

# --- 4. Create the Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# --- 5. Start Training! ---
print("\nStarting model training...")
trainer.train()
print("✅ Training complete.")

# --- 6. Manually Evaluate at the End ---
# Since it's not done automatically, we'll run evaluation now.
print("\nRunning final evaluation...")
eval_results = trainer.evaluate()
print("\n--- Final Model Performance ---")
print(eval_results)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1770567130.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting model training...


Step,Training Loss
500,0.184800


✅ Training complete.

Running final evaluation...



--- Final Model Performance ---
{'eval_loss': 0.0016249747714027762, 'eval_accuracy': 1.0, 'eval_f1-macro': 1.0, 'eval_runtime': 18.1042, 'eval_samples_per_second': 56.672, 'eval_steps_per_second': 3.59, 'epoch': 3.0}


In [12]:
# Cell 6: Final Evaluation
from sklearn.metrics import classification_report

print("Running final evaluation on the test set...")

# --- 1. Get Predictions ---
predictions = trainer.predict(tokenized_test_dataset)

# Get the raw predicted class IDs (0, 1, 2, or 3)
y_pred = np.argmax(predictions.predictions, axis=1)

# Get the true class IDs
y_true = tokenized_test_dataset["labels"]

# --- 2. Generate Classification Report ---
# Get the human-readable label names (e.g., "BENIGN", "DOS", etc.)
target_names = list(label2id.keys())

print("\n--- Final Classification Report ---")
report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
print(report)

print("\nThis report shows the precision, recall, and F1-score for each attack type.")
print("The 'macro avg' and 'weighted avg' give you the overall model performance.")

Running final evaluation on the test set...



--- Final Classification Report ---
              precision    recall  f1-score   support

      BENIGN     1.0000    1.0000    1.0000       309
 BRUTE_FORCE     1.0000    1.0000    1.0000       254
         DOS     1.0000    1.0000    1.0000       263
    PORTSCAN     1.0000    1.0000    1.0000       200

    accuracy                         1.0000      1026
   macro avg     1.0000    1.0000    1.0000      1026
weighted avg     1.0000    1.0000    1.0000      1026


This report shows the precision, recall, and F1-score for each attack type.
The 'macro avg' and 'weighted avg' give you the overall model performance.


In [13]:
# Cell 7: Save Your Trained Model
import os

model_save_path = "network_model"
trainer.save_model(model_save_path)

# Also save the tokenizer
tokenizer.save_pretrained(model_save_path)

print(f"✅ Model and tokenizer saved to: {model_save_path}")

# Let's see the files it created
print("\nFiles in the model directory:")
print(os.listdir(model_save_path))

✅ Model and tokenizer saved to: network_model

Files in the model directory:
['training_args.bin', 'special_tokens_map.json', 'model.safetensors', 'config.json', 'tokenizer.json', 'vocab.txt', 'tokenizer_config.json']


In [14]:
# Cell 8: Zip and Download Your Model
from google.colab import files
import shutil

zip_filename = "network_model.zip"
directory_to_zip = "network_model"

# Create a zip archive of the model directory
shutil.make_archive(directory_to_zip, 'zip', directory_to_zip)

print(f"Created {zip_filename}. Starting download...")

# Trigger the download
files.download(zip_filename)

Created network_model.zip. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>